<a href="https://colab.research.google.com/github/EllenKAlves/AuroraSinger_FIAP/blob/main/MGPEB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MGPEB — Módulo de Gerenciamento de Pouso e Estabilização de Base
# Missão: Aurora Siger | Destino: Marte

ETAPA 1 — Definição e Cadastro dos Módulos de Pouso

In [3]:
from collections import deque

In [13]:
# Cada módulo é um dicionário com os atributos básicos da missão. Estrutura: nome, tipo, prioridade (1=mais urgente), combustivel (%),massa (t), criticidade, horario_chegada_orbita

modulos_definidos = [
    {
        "nome": "Habitat Alpha",
        "tipo": "Habitação",
        "prioridade": 1,
        "combustivel": 85,
        "massa": 12.0,
        "criticidade": "Alta",
        "horario_chegada_orbita": "08:00"
    },
    {
        "nome": "Reator Solar",
        "tipo": "Energia",
        "prioridade": 2,
        "combustivel": 60,
        "massa": 8.5,
        "criticidade": "Alta",
        "horario_chegada_orbita": "09:30"
    },
    {
        "nome": "MedBase",
        "tipo": "Suporte Médico",
        "prioridade": 2,
        "combustivel": 90,
        "massa": 5.5,
        "criticidade": "Alta",
        "horario_chegada_orbita": "10:00"
    },
    {
        "nome": "BioLab",
        "tipo": "Laboratório Científico",
        "prioridade": 3,
        "combustivel": 72,
        "massa": 6.0,
        "criticidade": "Média",
        "horario_chegada_orbita": "11:00"
    },
    {
        "nome": "LogiStore",
        "tipo": "Logística",
        "prioridade": 4,
        "combustivel": 45,
        "massa": 15.0,
        "criticidade": "Baixa",
        "horario_chegada_orbita": "13:00"
    },
    {
        "nome": "TerraBot",
        "tipo": "Mineração",
        "prioridade": 5,
        "combustivel": 30,
        "massa": 18.0,
        "criticidade": "Baixa",
        "horario_chegada_orbita": "15:00"
    },
]

In [9]:
# Fila principal: módulos aguardando autorização de pouso (ordem de chegada)
fila_pouso = deque()

# Lista de módulos já pousados com sucesso
lista_pousados = []

# Lista de módulos em espera (pouso adiado)
lista_em_espera = []

# Lista de módulos em alerta (problema detectado)
lista_em_alerta = []

# Pilha (stack) de histórico de pousos: o último pouso bem-sucedido fica no topo (LIFO). Útil para auditoria rápida e rollback.
# Operações usadas: append() empilha no topo | pop() ou [-1] acessa o topo.
pilha_historico_pousos = []

In [10]:
# Cadastra todos os módulos na fila principal
for modulo in modulos_definidos:
    fila_pouso.append(modulo)

print("   🛸 MGPEB — Missão Aurora Siger inicializado")
print(f"\n📋 {len(fila_pouso)} módulos cadastrados na fila de pouso:\n")
for i, m in enumerate(fila_pouso, 1):
    print(f"  {i}. {m['nome']} | Tipo: {m['tipo']} | Prioridade: {m['prioridade']} | "
          f"Combustível: {m['combustivel']}% | Massa: {m['massa']}t | "
          f"Criticidade: {m['criticidade']} | Chegada: {m['horario_chegada_orbita']}")
print()

   🛸 MGPEB — Missão Aurora Siger inicializado

📋 6 módulos cadastrados na fila de pouso:

  1. Habitat Alpha | Tipo: Habitação | Prioridade: 1 | Combustível: 85% | Massa: 12.0t | Criticidade: Alta | Chegada: 08:00
  2. Reator Solar | Tipo: Energia | Prioridade: 2 | Combustível: 60% | Massa: 8.5t | Criticidade: Alta | Chegada: 09:30
  3. MedBase | Tipo: Suporte Médico | Prioridade: 2 | Combustível: 90% | Massa: 5.5t | Criticidade: Alta | Chegada: 10:00
  4. BioLab | Tipo: Laboratório Científico | Prioridade: 3 | Combustível: 72% | Massa: 6.0t | Criticidade: Média | Chegada: 11:00
  5. LogiStore | Tipo: Logística | Prioridade: 4 | Combustível: 45% | Massa: 15.0t | Criticidade: Baixa | Chegada: 13:00
  6. TerraBot | Tipo: Mineração | Prioridade: 5 | Combustível: 30% | Massa: 18.0t | Criticidade: Baixa | Chegada: 15:00



# ETAPA 2 — Regras de Decisão com Portas Lógicas



In [16]:
# Condições simuladas do ambiente marciano e da área de pouso
condicoes_ambiente = {
    "area_de_pouso_livre": True,       # Área disponível para receber o módulo
    "tempestade_de_areia": False,      # Tempestade ativa bloqueia qualquer pouso
    "sensores_operacionais": True,     # Sensores de navegação funcionando
    "comunicacao_terra": True,         # Link com controle na Terra ativo
    "reserva_emergencial": True       # Reserva de combustível para manobras críticas
}

In [17]:
# Função que avalia se o pouso de um módulo pode ser autorizado.

    #Regras booleanas aplicadas:
    # Combustível mínimo de 40% para garantir manobra de pouso segura
    # Área de pouso deve estar livre
    # Não pode haver tempestade de areia ativa
    # Sensores de navegação precisam estar operacionais
    # Módulos críticos exigem também comunicação ativa com a Terra

def autorizar_pouso(modulo, ambiente):
    motivos = []

    # Variáveis booleanas básicas
    combustivel_ok     = modulo["combustivel"] >= 40
    area_livre         = ambiente["area_de_pouso_livre"]
    sensores_ok        = ambiente["sensores_operacionais"]
    comunicacao_ok     = ambiente["comunicacao_terra"]
    reserva_ok         = ambiente["reserva_emergencial"]
    critico            = modulo["criticidade"] == "Alta"

    # Porta NOT
    # "sem tempestade" é a negação da flag de tempestade ativa
    sem_tempestade     = not ambiente["tempestade_de_areia"]

    # Porta OR
    # O combustível é considerado viável se estiver acima do mínimo OU se o módulo for crítico e houver reserva emergencial disponível.
    # (Módulos críticos têm direito a uma margem de manobra adicional.)
    combustivel_viavel = combustivel_ok or (critico and reserva_ok)

    # Porta AND principal
    # Todas as condições precisam ser verdadeiras simultaneamente
    condicao_basica = combustivel_viavel and area_livre and sem_tempestade and sensores_ok

    # AND adicional: módulos críticos também exigem comunicação com a Terra
    if critico:
        autorizado = condicao_basica and comunicacao_ok
    else:
        autorizado = condicao_basica

    # Registra os motivos de bloqueio para exibição
    if not combustivel_viavel:
        motivos.append(f"Combustível insuficiente ({modulo['combustivel']}% < 40%) e sem reserva emergencial")
    if not area_livre:
        motivos.append("Área de pouso ocupada")
    if not sem_tempestade:
        motivos.append("Tempestade de areia ativa")
    if not sensores_ok:
        motivos.append("Sensores de navegação inoperantes")
    if critico and not comunicacao_ok:
        motivos.append("Comunicação com a Terra perdida (módulo crítico)")

    return autorizado, motivos

In [18]:
modulo_teste = modulos_definidos[0]
autorizado, motivos = autorizar_pouso(modulo_teste, condicoes_ambiente)

print(f"Módulo: {modulo_teste['nome']}")
print(f"Autorizado: {autorizado}")
print(f"Motivos de bloqueio: {motivos}")

Módulo: Habitat Alpha
Autorizado: True
Motivos de bloqueio: []
